# 02 — Cleaning and Metadata Consolidation

## What this notebook is teaching

The goal of cleaning is not to make the table “look nice.” It is to create a **stable data contract** that later models can trust.

The raw scrape contains several kinds of uncertainty:

- repeated rows caused by resumable scraping;
- missing HTML fields;
- inconsistent Persian/Arabic characters;
- free-text education values;
- starter posts without normal post IDs;
- incomplete profile fields;
- exact duplicates and ID collisions.

The cleaning algorithm converts those irregular records into one consistent post-level table while preserving enough provenance to audit questionable cases.

### Main design principles

1. **Normalize representation, not meaning.** Persian character normalization fixes orthographic inconsistency; it does not rewrite the user's message.
2. **Never use a single weak identifier blindly.** Site post IDs are preferred when trustworthy, but fallback project IDs are constructed when necessary.
3. **Separate missing from zero whenever possible.** Missing metadata means “unknown,” not necessarily numerical zero.
4. **Audit destructive operations.** Deduplication can delete information, so conflicts are measured before rows are removed.
5. **Keep post content and profile/signature metadata conceptually separate.**

The methodological focus is on these data-engineering decisions rather than individual Pandas statements.


## Real-data input contract

The historical ~330 MB raw file is expected at:

`inputs/combined_all.csv`

and keep this configuration:

`"use_demo_if_input_missing": false`

With that setting, the notebook will **stop with an error** if the real input is missing instead of silently generating synthetic data.

Demo mode is retained only as an optional developer smoke test. It should not be used for report figures, metrics, or final artifacts.


In [2]:
from pathlib import Path
import sys, json
HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.common import load_json, set_seed
CFG = load_json(ROOT / "configs" / "project_config.json")
LEX = load_json(ROOT / "configs" / "lexicons.json")
set_seed(CFG["random_seed"])
print("Project root:", ROOT)
print("Project:", CFG["project_name"])

Project root: C:\Education\term6-private\DS\Project\MemberC_Early_Data_Pipeline_Project_Final_v2\MemberC_Early_Data_Pipeline_Project_Final_v2
Project: Member C Early Data Pipeline: Scraping, Cleaning, Proxy Construction, and Initial Label Sampling


In [3]:
import pandas as pd
from src.cleaning import load_raw_posts, clean_dataset
CCFG=CFG["cleaning"]

## Important text-preservation detail

The original cleaning notebook used the raw `content` column for the structural
filters (non-empty and at least three whitespace-separated words) and did **not**
replace the whole column with Hazm-normalized text afterward.

That matters because the later proxy notebook computed character, punctuation,
and emoji counts from the stored raw text. Hazm normalization is therefore done
locally only inside tokenization/stemming functions, matching the historical
workflow.

`thread_url` is removed during cleaning. `profile_url` is retained only long
enough to replay the historical gender enrichment, then removed before the final
cleaned dataset is written.


## Historical category/sub-category repair — cached, not repeated manually

Three historical scrape files were collected without `category` and `sub_category`
columns. Their forum origin was still known from the source filename, so during the
real development run we manually assigned one label pair to each source:

- `ninisite_others_others_dataset.csv` → `متفرقه / عمومی`
- `ninisite_others_pregnancy_dataset.csv` → `دوران-بارداری / سایر-موضوعات-در-بارداری`
- `ninisite_wedding_marriage_dataset.csv` → `ازدواج-و-شروع-زندگی / عروسی-و-زندگی-مشترک`

Rows were matched back into the cleaned table with the historical key:

`thread_id + author + posted_at`

The original execution repaired **21,001 + 37,517 + 40,924 = 99,442 rows** and
ended with zero missing categories.

The finalized reproduction does not require those three large CSVs because their completed mapping is frozen. Their completed
mapping is frozen in:

`inputs/enrichment/category_recovery.csv`

Notebook 02 only replays that cache and records `category_source`. It makes no
network request and does not ask the user to manually re-enter labels.


## 1. Load the historical raw corpus

For the original project, the preferred input is the already-combined raw scrape:

`inputs/combined_all.csv`

This is the large (~330 MB) table produced after the scraping/recovery stages. It already contains post fields, profile metadata, category, and sub-category columns.

### Why use `combined_all.csv` here instead of the scraper checkpoints?

If this file exists, it is the safest historical reproduction input because it prevents us from concatenating old checkpoint fragments a second time and accidentally recreating the duplication problem that occurred during development.

The expected columns include:

`thread_id, thread_title, post_id, author, profile_url, posted_at, content, likes, reply_to, is_starter, user_post_count, gender, gender_code, join_date, user_age, education, status, children_count, signature, thread_url, category, sub_category`

The loader therefore follows this priority:

1. if `inputs/combined_all.csv` exists, load it directly;
2. otherwise, reconstruct from `data/raw/posts/*.csv`;
3. if neither exists and demo mode is enabled, generate a tiny synthetic smoke-test table.

The 330 MB raw CSV should **not** be committed into the project ZIP or Git repository. Keep it locally under `inputs/`.

The code also recognizes the historical column name `user_age` and converts it into the cleaned numeric `age_num` feature.


In [4]:
raw, sources = load_raw_posts(
    ROOT,
    CCFG["raw_posts_glob"],
    use_demo=CFG.get("use_demo_if_input_missing", False),
    combined_all_file=CCFG.get("combined_all_file"),
    prefer_combined_all=CCFG.get("prefer_combined_all", True),
)
DEMO_MODE = sources == ["DEMO_GENERATED"]

print("Expected historical input:", ROOT / CCFG["combined_all_file"])
print("Sources:", sources)
print("Raw rows:", len(raw), "DEMO_MODE:", DEMO_MODE)

if DEMO_MODE:
    print(
        "\nWARNING: synthetic DEMO data is being used. "
        "Do not use these outputs in the report."
    )
else:
    print("\nREAL DATA MODE: combined/raw Ninisite data loaded.")

display(raw.head())


Expected historical input: C:\Education\term6-private\DS\Project\MemberC_Early_Data_Pipeline_Project_Final_v2\MemberC_Early_Data_Pipeline_Project_Final_v2\inputs\combined_all.csv
Sources: ['inputs\\combined_all.csv']
Raw rows: 414994 DEMO_MODE: False

REAL DATA MODE: combined/raw Ninisite data loaded.


,thread_id,thread_title,post_id,author,profile_url,posted_at,content,likes,reply_to,is_starter,...,gender_code,join_date,user_age,education,status,children_count,signature,thread_url,category,sub_category
0,17006577,مادرشوهر شمام فرق میزاره بینتون,NaN,هانيا,https://www.ninisite.com/user/2b3ac517-4251-48...,1404/06/27 13:29,من چند ساله عروس این خانواده شدم\nهرسال جلو چش...,0,NaN,True,...,1,1400/02/02,29 سال,ليسانس,NaN,0,NaN,https://www.ninisite.com/discussion/topic/1700...,دوران-بارداری,بارداری-ناخواسته-سقط-و-واکنشهای-روحی-و-روانی-ا...
1,17006577,مادرشوهر شمام فرق میزاره بینتون,395597169.0,zhoooan,https://www.ninisite.com/user/85d65c46-9f49-41...,1404/06/27 13:30,یعنی به جاریتون تعارف میزنن و با خودشون میبرن ...,0,NaN,False,...,1,1396/07/22,NaN,درج نشده است,من و فرزندانم,1,NaN,https://www.ninisite.com/discussion/topic/1700...,دوران-بارداری,بارداری-ناخواسته-سقط-و-واکنشهای-روحی-و-روانی-ا...
2,17006577,مادرشوهر شمام فرق میزاره بینتون,395597264.0,هانيا,https://www.ninisite.com/user/2b3ac517-4251-48...,1404/06/27 13:32,بله با جاری خواهرشوهرم اینا میرن کلا,0,395597169.0,False,...,1,1400/02/02,29 سال,ليسانس,NaN,0,NaN,https://www.ninisite.com/discussion/topic/1700...,دوران-بارداری,بارداری-ناخواسته-سقط-و-واکنشهای-روحی-و-روانی-ا...
3,17006577,مادرشوهر شمام فرق میزاره بینتون,395597694.0,zhoooan,https://www.ninisite.com/user/85d65c46-9f49-41...,1404/06/27 13:37,شما هم مسئولیت قبول نکن بگو ما هم می‌خوایم بری...,1,395597264.0,False,...,1,1396/07/22,NaN,درج نشده است,من و فرزندانم,1,NaN,https://www.ninisite.com/discussion/topic/1700...,دوران-بارداری,بارداری-ناخواسته-سقط-و-واکنشهای-روحی-و-روانی-ا...
4,17006577,مادرشوهر شمام فرق میزاره بینتون,395597716.0,بهاره_ریحانه,https://www.ninisite.com/user/bc8049a6-f884-4f...,1404/06/27 13:37,شاید به شوهرت میگن اون مخالفت میکنه و میگه ما ...,0,NaN,False,...,1,1402/01/14,NaN,NaN,من و فرزندانم,1,NaN,https://www.ninisite.com/discussion/topic/1700...,دوران-بارداری,بارداری-ناخواسته-سقط-و-واکنشهای-روحی-و-روانی-ا...


In [5]:
category_cache = ROOT / CCFG["category_recovery_csv"]
male_cache = ROOT / CCFG["historical_male_profile_cache_pkl"]

print("Category recovery cache:", category_cache, "exists:", category_cache.exists())
print("Historical male-profile cache:", male_cache, "exists:", male_cache.exists())

if not category_cache.exists():
    raise FileNotFoundError("Frozen category_recovery.csv is missing.")
if not male_cache.exists():
    raise FileNotFoundError("Frozen historical_male_profile_cache.pkl is missing.")

print("Cleaning recovery mode: cached artifacts only; no web requests.")


Category recovery cache: C:\Education\term6-private\DS\Project\MemberC_Early_Data_Pipeline_Project_Final_v2\MemberC_Early_Data_Pipeline_Project_Final_v2\inputs\enrichment\category_recovery.csv exists: True
Historical male-profile cache: C:\Education\term6-private\DS\Project\MemberC_Early_Data_Pipeline_Project_Final_v2\MemberC_Early_Data_Pipeline_Project_Final_v2\inputs\enrichment\historical_male_profile_cache.pkl exists: True
Cleaning recovery mode: cached artifacts only; no web requests.


## 2. Cleaning pipeline: order matters

The transformations are applied in a deliberate order.

### 2.1 Remove structurally unusable records

Rows with no meaningful author/content and posts shorter than the configured minimum are removed before expensive feature construction. This prevents obvious noise from propagating into later statistics.

### 2.2 Normalize Persian text

Hazm-style normalization reduces equivalent character variants such as Arabic/Persian forms of ی and ک, whitespace irregularities, and half-space inconsistencies.

The purpose is **canonical representation**. Two visually equivalent strings should not appear different merely because of Unicode variation.

### 2.3 Recover and standardize metadata

Age is converted into a numeric representation where possible.

Education is a free-text field, so an ordered rule system maps spelling variants and synonyms into a smaller categorical vocabulary. Rule ordering matters: a specific phrase such as “فوق دیپلم” must be recognized before the more general substring “دیپلم”.

### 2.4 Merge profile information

Profile metadata is joined back to posts through stable keys/profile URLs. The join enriches a post; it must not multiply the number of posts.

### 2.5 Construct robust post identity

Preferred identity hierarchy:

1. reliable site `post_id`;
2. deterministic starter-post ID based on `thread_id`;
3. fallback combination using thread/author/time;
4. collision-safe suffix if two genuinely different records still map to the same project ID.

This is more reliable than assuming every starter post has a native post ID.

### 2.6 Build signature features

The user signature is not merged into post text. Instead, counts such as signature length, punctuation, emoji, and positive/negative lexical counts are extracted as separate structured features.

This prevents a permanent profile signature from being confused with the language of a specific post.

The code implementing these rules lives in `src/cleaning.py`; the notebook is the readable orchestration layer.


In [6]:
profiles_path=ROOT/CCFG["profile_checkpoint"]
audit_path=ROOT/CCFG["audit_json"]
cleaned,audit=clean_dataset(
    raw,
    profiles_path,
    CCFG,
    LEX,
    audit_path,
    project_root=ROOT,
)
print(json.dumps(audit, ensure_ascii=False, indent=2))
display(cleaned.head())

required_provenance = {'category_source', 'gender_source'}
missing_provenance = required_provenance - set(cleaned.columns)
if missing_provenance:
    raise RuntimeError(
        f'Missing provenance columns after cleaning: {sorted(missing_provenance)}. '
        'This usually means an old src/cleaning.py was imported or project_root was not passed.'
    )
print('Provenance columns present:', sorted(required_provenance))

# Historical numeric site IDs should never retain a pandas float suffix.
float_like_ids = cleaned["unique_post_id"].astype(str).str.fullmatch(r"\d+\.0+")
assert not float_like_ids.any(), (
    f"{int(float_like_ids.sum())} unique_post_id values still have terminal .0"
)
print("unique_post_id numeric canonicalization: PASS")


{
  "input_rows": 414994,
  "persian_backend": "hazm",
  "after_nonempty_content": 413180,
  "after_raw_three_word_filter": 362026,
  "after_optional_author_filter": 362026,
  "content_globally_normalized": false,
  "category_repair": {
    "mode": "cached_historical_replay",
    "network_requests": 0,
    "cache_loaded": true,
    "updated_rows": 99442,
    "remaining_missing_category": 0,
    "remaining_missing_sub_category": 0,
    "cache_rows": 108994,
    "filled_category_values": 99442,
    "filled_sub_category_values": 99442,
    "expected_historical_updates": 99442
  },
  "missing_thread_title_before_drop": 9,
  "after_title_filter": 362017,
  "gender_code_before_recovery": {
    "1": 335182,
    "-2": 19117,
    "-1": 7718
  },
  "gender_recovery": {
    "mode": "historical_male_profile_cache_replay",
    "network_requests": 0,
    "male_cache_loaded": true,
    "male_cache_profiles": 1437,
    "rows_marked_male_from_cache": 6535,
    "remaining_assumed_female_rows": 20300,
  

,thread_id,thread_title,post_id,author,posted_at,content,likes,reply_to,is_starter,user_post_count,...,sig_char_count,sig_punct_count,sig_question_count,sig_excl_count,sig_emoji_count,sig_word_count,sig_neg_count,sig_pos_count,sig_pos_emoji,sig_neg_emoji
0,17006577,مادرشوهر شمام فرق میزاره بینتون,NaN,هانيا,1404/06/27 13:29,من چند ساله عروس این خانواده شدم\nهرسال جلو چش...,0,NaN,True,969.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
1,17005131,همیشه فکر میکردم هرکسی همسر کیی ک دوستش داشتم ...,NaN,جهان_هستی2,1404/06/27 03:14,ولی بعد مدت ها عشق و عاشقی تازه چشمام باز شده ...,0,NaN,True,15987.0,...,431,2,0,0,2,84,0,4,2,0
2,17008674,کسی درباره شکایت از مزاحم تلفن چیزی میدونه,NaN,ستاره_ههه,1404/06/27 20:08,یکی از پارساله ولم نمیکنه با خطای مختلف زنگ و ...,0,NaN,True,2855.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
3,17225378,سن بارداری,NaN,mmmary66,1404/07/30 07:18,دوستان الان که سن بارداری بالا رفته ما پدر و م...,0,NaN,True,2143.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
4,17448188,ساک جنین بدون جنین و مایع زرد,NaN,khanomenon,1404/09/08 00:43,دوهفته پیش رفتم سونو چون یادم رفته بود کی پریو...,0,NaN,True,185.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


Provenance columns present: ['category_source', 'gender_source']
unique_post_id numeric canonicalization: PASS


## Gender repair — replay the historical result without re-scraping

The historical project scraped profile gender, corrected an early marker-parser
mistake, and rechecked unresolved profiles. The historical notebook recorded
**6,932 scraped profiles**, including exactly **1,437 male profiles**.

The complete checkpoint is no longer available. For reproducibility only, the
final male profile URLs were reconstructed once from the historical final dataset
and `combined_all.csv`. The reconstructed set also contains exactly **1,437**
unique male profiles, matching the original notebook.

Notebook 02 makes **zero network requests**. It loads that frozen male-profile
cache, marks matching profiles as male, preserves explicit female observations,
then applies the historical decision that the remaining unresolved users are
female after the male-recovery/rechecking step. `gender_source` records provenance.

The derived cache is a replay artifact, not a new scrape.


In [7]:
print("Historical cleaning checkpoints")
print("  expected raw rows:                 414994")
print("  expected non-empty content:        413180")
print("  expected >=3 raw-word rows:        362026")
print("  expected final cleaned rows:       362017")
print("  actual output rows:                ", len(cleaned))

if len(raw) == 414994 and len(cleaned) != 362017:
    print("WARNING: final row count does not reproduce the historical 362,017-row corpus.")
else:
    print("Row-count checkpoint is consistent with the historical run.")

print("\nGender after cleaning:")
print(cleaned['gender_code'].value_counts(dropna=False))
if (cleaned['gender_code'] == 0).sum() == 0:
    print("WARNING: no male rows are present. Historical cleaning recovered male profiles, so do not interpret this as a true all-female corpus.")

missing_cat=(cleaned['category'].isna() | cleaned['category'].astype(str).str.strip().eq('')).sum()
print("\nMissing category rows:", missing_cat)
if missing_cat:
    print("NOTE: provide the three historical auxiliary category files listed in inputs/README.md to reproduce category completion.")


Historical cleaning checkpoints
  expected raw rows:                 414994
  expected non-empty content:        413180
  expected >=3 raw-word rows:        362026
  expected final cleaned rows:       362017
  actual output rows:                 362017
Row-count checkpoint is consistent with the historical run.

Gender after cleaning:
gender_code
1    355482
0      6535
Name: count, dtype: int64

Missing category rows: 0


## 3. Duplicate-ID safety check

Deduplication is one of the riskiest cleaning operations because it is irreversible.

The historical notebooks showed that a naïve rule such as:

`drop every repeated unique_post_id`

could remove rows that shared an identifier but had different content.

Therefore the algorithm distinguishes:

- **true duplicates:** repeated records representing the same post;
- **ID conflicts:** same apparent site ID but materially different records.

True duplicates may be collapsed. Conflicts are retained and assigned collision-safe project identifiers.

### Invariant

After cleaning:

`number of unique_post_id values == number of rows`

This is essential because every later annotation, fold assignment, model prediction, and fusion join uses `unique_post_id`.


In [8]:
assert cleaned["unique_post_id"].is_unique
print("Guaranteed unique IDs:", cleaned["unique_post_id"].nunique(), "/", len(cleaned))
print("Conflicting site-post IDs retained for audit:", audit.get("conflicting_post_ids"))

Guaranteed unique IDs: 362017 / 362017
Conflicting site-post IDs retained for audit: 329


## 4. Missing gender: explicit assumption, not hidden truth

Some user profiles did not expose a usable gender value even after selective re-scraping.

The historical pipeline eventually imputed unresolved values as female because the forum population was overwhelmingly female and manual inspection supported that operational choice.

This notebook keeps that assumption **visible in configuration** rather than burying it inside code.

Why is that important?

- an imputed value is not an observed fact;
- later analysts can rerun a sensitivity analysis using `unknown`;
- demographic assumptions can affect model behavior and should be documented.

This variable is therefore treated as ordinary metadata with a known limitation, not as ground truth.


In [9]:
print("Category distribution (top values):")
print(cleaned["category"].value_counts(dropna=False).head(20))
print("\nMissing category:", cleaned["category"].isna().sum())
print("Missing sub_category:", cleaned["sub_category"].isna().sum())
print("Category provenance:")
print(cleaned["category_source"].value_counts(dropna=False))
print("\nCategory recovery audit:")
print(json.dumps(audit.get("category_repair", {}), ensure_ascii=False, indent=2))

print("\nGender distribution:")
print(cleaned["gender"].value_counts(dropna=False))
print("\nGender provenance:")
print(cleaned["gender_source"].value_counts(dropna=False))
print("\nGender recovery audit:")
print(json.dumps(audit.get("gender_recovery", {}), ensure_ascii=False, indent=2))

assert cleaned["category"].notna().all()
assert cleaned["sub_category"].notna().all()
assert pd.to_numeric(cleaned["gender_code"], errors="coerce").isin([0, 1]).all()
assert audit.get("gender_recovery", {}).get("network_requests") == 0
assert audit.get("category_repair", {}).get("network_requests") == 0
print("\nConfirmed: category and gender recovery are complete and Cleaning made no recovery network requests.")


Category distribution (top values):
category
دوران-بارداری          222431
بارداری-و-زایمان        77661
ازدواج-و-شروع-زندگی     40924
متفرقه                  21001
Name: count, dtype: int64

Missing category: 0
Missing sub_category: 0
Category provenance:
category_source
original_combined_all                   262575
cached_historical_source_file_repair     99442
Name: count, dtype: int64

Category recovery audit:
{
  "mode": "cached_historical_replay",
  "network_requests": 0,
  "cache_loaded": true,
  "updated_rows": 99442,
  "remaining_missing_category": 0,
  "remaining_missing_sub_category": 0,
  "cache_rows": 108994,
  "filled_category_values": 99442,
  "filled_sub_category_values": 99442,
  "expected_historical_updates": 99442
}

Gender distribution:
gender
زن     355482
مرد      6535
Name: count, dtype: int64

Gender provenance:
gender_source
original_female                                                  335182
remaining_unresolved_assumed_female_after_historical_recovery    

## 5. Save the cleaned data contract

The cleaned table is the handoff from data engineering to feature/proxy construction.

At minimum, downstream stages need:

- a guaranteed `unique_post_id`;
- normalized `content`;
- thread/author/time information;
- `is_starter` for auditing;
- cleaned categorical/profile fields.

The save step is deliberately separated from proxy construction. This means the same cleaned corpus can be reused for different modeling ideas without repeating web scraping or changing the raw data.


In [10]:
out=ROOT/CCFG["output_csv"]; out.parent.mkdir(parents=True,exist_ok=True)
cleaned.to_csv(out,index=False,encoding="utf-8-sig")
marker=out.parent/"DEMO_MODE.marker"
if DEMO_MODE:
    marker.write_text("Synthetic smoke-test data; replace with the real cleaned corpus.\n",encoding="utf-8")
elif marker.exists():
    marker.unlink()
print("Saved:", out, "rows:", len(cleaned), "DEMO_MODE:", DEMO_MODE)

Saved: C:\Education\term6-private\DS\Project\MemberC_Early_Data_Pipeline_Project_Final_v2\MemberC_Early_Data_Pipeline_Project_Final_v2\data\processed\ninisite_cleaned_enriched.csv rows: 362017 DEMO_MODE: False


## 6. Final assertions: fail loudly

Assertions are executable documentation.

Instead of assuming the cleaning succeeded, the notebook checks critical invariants:

- required columns exist;
- every project post ID is unique.

A pipeline should stop when a contract is broken rather than silently produce a corrupted downstream dataset.

### System-designer takeaway

The cleaning stage is:

**filter → normalize → recover metadata → standardize categories → establish identity → audit duplicates → derive separate metadata features → validate contract.**

The key learning objective is understanding why each transformation protects downstream validity.


In [11]:
assert "profile_url" not in cleaned.columns
assert "thread_url" not in cleaned.columns
assert audit.get("content_globally_normalized") is False
print("Raw content preservation: PASS")
print("URL columns removed after enrichment: PASS")
required=["unique_post_id","content","thread_id","author","posted_at","is_starter","education_clean","age_num"]
missing=[c for c in required if c not in cleaned]
assert not missing, missing
assert cleaned["unique_post_id"].is_unique
print("CLEANING NOTEBOOK COMPLETED SUCCESSFULLY")


Raw content preservation: PASS
URL columns removed after enrichment: PASS
CLEANING NOTEBOOK COMPLETED SUCCESSFULLY
